# 04 - NLP Text Preprocessing

**Goal**: Transform raw job descriptions into clean, normalized text ready for NLP models.

**Learning concepts**: HTML cleaning, text normalization, lemmatization vs stemming,
stopword removal, text statistics, spaCy NLP pipeline.

**Why this matters**: ML models can't read raw text — they need numbers.
But before we convert text to numbers (Phase 2b: embeddings), we need to
clean it. Garbage in → garbage out.

**Pipeline**: Raw description → strip HTML → normalize whitespace → lowercase → lemmatize → save

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from talentlens.config import POSTINGS_CLEAN_PARQUET, POSTINGS_NLP_PARQUET, PROCESSED_DIR
from talentlens.features import clean_text, add_text_stats, lemmatize_texts
from talentlens.plots import plot_distribution, save_fig

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
%matplotlib inline

In [ ]:
# Load the cleaned dataset from Phase 1
if not POSTINGS_CLEAN_PARQUET.exists():
    raise FileNotFoundError(
        "postings_clean.parquet not found! "
        "Run notebook 02-mp-data-cleaning.ipynb first."
    )

df = pd.read_parquet(POSTINGS_CLEAN_PARQUET)
print(f"Loaded {len(df):,} cleaned postings")
df.info()

## 1. Inspect Raw Descriptions

Before cleaning, let's look at what we're working with. Job descriptions
on LinkedIn often contain HTML formatting (bold, lists, links) that we need to strip.

In [ ]:
# Look at a few raw descriptions to see what we're dealing with
print("=== Sample raw description (first 500 chars) ===")
print(df['description'].iloc[0][:500])
print("\n" + "="*60)
print("\n=== Another sample ===")
print(df['description'].iloc[100][:500])

In [ ]:
# How long are descriptions before cleaning?
raw_lengths = df['description'].str.len()
print("Raw description length (characters):")
print(raw_lengths.describe().round(0))

# Check for HTML content
has_html = df['description'].str.contains('<', na=False).sum()
print(f"\nDescriptions containing '<' (likely HTML): {has_html:,} ({has_html/len(df)*100:.1f}%)")

### What to notice

- Many descriptions contain HTML tags (`<p>`, `<li>`, `<br>`, `<strong>`, etc.)
- Some have encoded entities (`&amp;`, `&nbsp;`)
- Lengths vary widely — some are short blurbs, others are walls of text
- This raw text would confuse NLP models — HTML tags aren't "words" the model should learn from

## 2. Clean Text

Our `clean_text()` function does 4 things in order:
1. **Decode HTML entities** (`&amp;` → `&`)
2. **Strip HTML tags** (`<p>text</p>` → `text`)
3. **Remove URLs** (not useful for NLP)
4. **Normalize whitespace + lowercase**

We apply this to the `description` column to create `desc_clean`.

In [ ]:
from tqdm import tqdm
tqdm.pandas(desc="Cleaning text")

# Apply clean_text to every description
# .progress_apply shows a progress bar (same as .apply but with tqdm)
df['desc_clean'] = df['description'].progress_apply(clean_text)

# Before vs after comparison
print("=== BEFORE cleaning (first 300 chars) ===")
print(df['description'].iloc[0][:300])
print("\n=== AFTER cleaning (first 300 chars) ===")
print(df['desc_clean'].iloc[0][:300])

In [ ]:
# How did cleaning affect text length?
clean_lengths = df['desc_clean'].str.len()
print("Clean description length (characters):")
print(clean_lengths.describe().round(0))

reduction = (1 - clean_lengths.mean() / raw_lengths.mean()) * 100
print(f"\nAverage length reduction: {reduction:.1f}% (HTML + URLs removed)")

## 3. Text Statistics

Before going deeper with NLP, let's add simple numeric features from the text:
- **Word count**: How verbose is the description? (Longer descriptions may correlate with salary or engagement)
- **Character count**: Raw size of the text
- **Title word count**: Short vs long job titles

These are "free" features — no ML needed, just counting.

In [ ]:
df = add_text_stats(df)

print("\nDescription word count stats:")
print(df['desc_word_count'].describe().round(0))

print("\nTitle word count stats:")
print(df['title_word_count'].describe().round(0))

In [ ]:
# Visualize description length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['desc_word_count'].clip(0, 1000), bins=50, ax=axes[0])
axes[0].set_title('Description Word Count Distribution')
axes[0].set_xlabel('Word Count')
axes[0].axvline(df['desc_word_count'].median(), color='red', linestyle='--', label=f"Median: {df['desc_word_count'].median():.0f}")
axes[0].legend()

sns.histplot(df['title_word_count'].clip(0, 15), bins=15, ax=axes[1])
axes[1].set_title('Title Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].axvline(df['title_word_count'].median(), color='red', linestyle='--', label=f"Median: {df['title_word_count'].median():.0f}")
axes[1].legend()

plt.tight_layout()
save_fig(fig, 'nlp_text_length_distributions')
plt.show()

### Interpretation

- **Description word count**: Most job descriptions are in the 100-500 word range. Very short descriptions (<50 words) may be low-effort postings. Very long ones (>800 words) may be from companies with detailed templates.
- **Title word count**: Most titles are 2-5 words ("Software Engineer", "Senior Data Scientist"). Unusually long titles (8+) may include location or department info.

## 4. Lemmatization with spaCy

**What is lemmatization?** Converting words to their base form:
- "running" → "run"
- "engineers" → "engineer"
- "better" → "good"

**Why?** So that "engineering", "engineers", and "engineer" are all treated as the same concept by ML models.

**Lemmatization vs Stemming**:
- **Stemming** (NLTK): crude suffix stripping → "running" → "run", but "university" → "univers" (broken!)
- **Lemmatization** (spaCy): uses vocabulary + grammar → always produces real words
- We use spaCy because real words are important for topic modeling and embeddings later.

**We also remove stopwords** ("the", "is", "and", etc.) — these are so common they add noise, not signal.

**Performance note**: We use `nlp.pipe()` for batch processing. Processing 123K texts one at a time would take hours; batching makes it manageable (minutes).

In [ ]:
# Lemmatize the cleaned descriptions
# This will take a few minutes on 123K texts — the progress bar shows where we are
df['desc_lemmatized'] = lemmatize_texts(df['desc_clean'], batch_size=2000)

In [ ]:
# Before vs after lemmatization
print("=== CLEANED text (first 200 chars) ===")
print(df['desc_clean'].iloc[0][:200])
print("\n=== LEMMATIZED text (first 200 chars) ===")
print(df['desc_lemmatized'].iloc[0][:200])

# How much shorter is the lemmatized text? (stopword removal shrinks it)
lemma_word_count = df['desc_lemmatized'].str.split().str.len()
reduction = (1 - lemma_word_count.mean() / df['desc_word_count'].mean()) * 100
print(f"\nWord count reduction after lemmatization + stopword removal: {reduction:.1f}%")
print(f"Median words: {df['desc_word_count'].median():.0f} (original) → {lemma_word_count.median():.0f} (lemmatized)")

### What happened

- Stopwords ("the", "is", "a", "and", "to", "of", "in", ...) were removed — these make up ~40-50% of English text
- Words were lemmatized: "responsibilities" → "responsibility", "managing" → "manage"
- The remaining words are the **content words** — the ones that actually distinguish one job from another
- This is what we'll feed into TF-IDF (notebook 05) to find the most important terms per job category

## 5. Save Preprocessed Data

Save the enriched DataFrame with all new columns so notebook 05 and 06 can load it directly.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(POSTINGS_NLP_PARQUET, index=False)

print(f"Saved to {POSTINGS_NLP_PARQUET}")
print(f"Shape: {df.shape}")
print(f"File size: {POSTINGS_NLP_PARQUET.stat().st_size / 1e6:.1f} MB")
print(f"\nNew columns added in this notebook:")
new_cols = ['desc_clean', 'desc_word_count', 'desc_char_count', 'title_word_count', 'desc_lemmatized']
for col in new_cols:
    print(f"  - {col}")

## Summary

### What we did
1. Inspected raw descriptions — found HTML tags, entities, URLs
2. Cleaned text — stripped HTML, normalized whitespace, lowercased
3. Added text statistics — word counts and character counts as simple features
4. Lemmatized with spaCy — reduced words to base forms, removed stopwords
5. Saved enriched dataset as `postings_nlp.parquet`

### Mental model: The NLP preprocessing pipeline

```
Raw HTML text → clean_text() → desc_clean (lowercase, no HTML)
                                    ↓
                             lemmatize_texts() → desc_lemmatized (base words, no stopwords)
                                    ↓
                          Ready for: TF-IDF, sentiment, embeddings
```

### Key decisions
- **spaCy over NLTK**: Real lemmas (not broken stems), faster batch processing
- **Disabled NER/parser**: 3x speed improvement — we only need tokenizer + lemmatizer
- **Kept original description**: `desc_clean` for human reading, `desc_lemmatized` for ML

---

**→ Next**: Notebook 05 — Feature Engineering (TF-IDF, sentiment, entry-level paradox)